# Plan your trip with Kayak
**Auteur :** Samer MECHHOUD
<br>**Objectif :** Recommander les 5 meilleures destinations françaises et les 20 meilleurs hôtels associés, en se basant sur des données météo (API OpenWeatherMap) et des données hôtelières (scraping Booking.com).

---

**Pipeline :**
`Collecte de données(Géocodage + API Météo + Scraping Hôtels)` -> `Data Lake` ->`Transformation, Nettoyage & Scoring` -> `Data Warehouse` -> `Visualisation`

Ce projet met en oeuvre un pipeline de collecte de donnée et ETL complet :

| Etape | Technologie |
|---|---|
| Géocodage des destinations | Nominatim (OpenStreetMap) |
| Données météo (agrégées sur l'horizon de prevision - 5 jours) | API OpenWeatherMap |
| Scraping hôtels | Selenium |
| Stockage brut | AWS S3 (Data Lake) |
| Extract & Transform | Python depuis le S3 |
| Load | AWS Redshift (Data Warehouse) |
| Visualisation | SQLAlchemy / Plotly Express (Mapbox) |

## 1. Imports

In [20]:
# Bibliothèques standard Python
import json, time, re, os
from io import StringIO
from pathlib import Path
from datetime import date, timedelta
import random

# Manipulation de données
import pandas as pd      # DataFrames : lecture, transformation, export CSV
import numpy as np       # Calculs numériques

# Requêtes HTTP
import requests          # Appels API REST

# Visualisation interactive
import plotly.express as px  # Cartes Mapbox interactives

# Web Scraping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import undetected_chromedriver as uc

# AWS SDK
import boto3
from sqlalchemy import create_engine
from sqlalchemy.types import VARCHAR
from sqlalchemy.sql import text

# Charger les vaiables d'environnement depuis .env
from dotenv import load_dotenv
load_dotenv()

# Dossier de sortie des données collectées (le créer s'il n'existe pas) --> pour le test et debug
DATA = Path("data")
DATA.mkdir(exist_ok=True)


## 2. Géocodage des 35 destinations (API Nominatim)
Récupération des coordonnées GPS via l'API Nominatim qui est basée sur OpenStreetMap.

La liste reprend exactement les 35 villes demandées.

In [2]:
# Liste officielle des 35 destinations attendues par Kayak
villes = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris",
    "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar",
    "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon",
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes",
    "Saintes Maries de la Mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

url_geocod = "https://nominatim.openstreetmap.org/search"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
villes_data = []

for ville in villes:
    try:
        # On precise "France" pour éviter les homonymes
        params = {
            'q': ville + ', France',
            'format': 'json',
            'limit': 1
        }
        response = requests.get(url_geocod, params=params, headers=headers)
        if 200 == response.status_code:
            villes_data.append({
                "city": ville,
                "latitude": response.json()[0]['lat'],
                "longitude": response.json()[0]['lon']
            })
            print(f"{ville} - trouvée")
        else:
            print(f"{ville} - non trouvée")

        # Pause obligatoire : Nominatim impose 1 requête/seconde au maximum --> on attend 1.5 seconde pour lancer la prochaine requête
        time.sleep(1.5)
    except Exception as e:
        print(f"Erreur {ville}: {e}")

# Création du DataFrame et ajout d'un identifiant unique par ville
df_villes = pd.DataFrame(villes_data)
df_villes.insert(0, "city_id", range(1, len(df_villes) + 1))
df_villes['latitude'] = df_villes['latitude'].astype(dtype='float')
df_villes['longitude'] = df_villes['longitude'].astype(dtype='float')

# Sauvegarde locale des coordonnées gps des destinations
df_villes.to_csv(DATA/"villes_gps.csv", index=False)

print(f"\nGeocodage terminé : {len(df_villes)}/35 villes")
display(df_villes.head())

Mont Saint Michel - trouvée
St Malo - trouvée
Bayeux - trouvée
Le Havre - trouvée
Rouen - trouvée
Paris - trouvée
Amiens - trouvée
Lille - trouvée
Strasbourg - trouvée
Chateau du Haut Koenigsbourg - trouvée
Colmar - trouvée
Eguisheim - trouvée
Besancon - trouvée
Dijon - trouvée
Annecy - trouvée
Grenoble - trouvée
Lyon - trouvée
Gorges du Verdon - trouvée
Bormes les Mimosas - trouvée
Cassis - trouvée
Marseille - trouvée
Aix en Provence - trouvée
Avignon - trouvée
Uzes - trouvée
Nimes - trouvée
Aigues Mortes - trouvée
Saintes Maries de la Mer - trouvée
Collioure - trouvée
Carcassonne - trouvée
Ariege - trouvée
Toulouse - trouvée
Montauban - trouvée
Biarritz - trouvée
Bayonne - trouvée
La Rochelle - trouvée

Geocodage terminé : 35/35 villes


,city_id,city,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


## 3. Données météo : API OpenWeatherMap
Nécéssité d'avoir une clé pour requêter.

Endpoint : `data/2.5/forecast` (prévisions sur 5 jours avec un step de 3 h, gratuit).
<br>On récupère les prévisions météo sur 5 jours, l'offre de l'API pour avoir des présvisions sur plus de 5 jours est devenue payante.

In [3]:
# La clé de l'API est chargé dans une variable d'environnement
API_KEY = os.getenv("OPENWEATHER_API_KEY")

weather_results = []

for _, row in df_villes.iterrows():
    # units=metric -> températures en ° Celsius
    # lang=fr -> descriptions en français
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        'appid': API_KEY,
        'units': 'metric',
        'lang': 'fr',
        'lat': row['latitude'],
        'lon': row['longitude']  
    }
    response = requests.get(url, params=params)

    if 200 == response.status_code:
        previsions = response.json()["list"]   # 40 créneaux : 5 jours / 3 h

        # Agrégation sur l'énsembe de la période
        temps = [pred["main"]["temp"] for pred in previsions]
        temps_feels = [pred["main"]["feels_like"] for pred in previsions]
        temps_min = [pred["main"]["temp_min"] for pred in previsions]
        temps_max = [pred["main"]["temp_max"] for pred in previsions]
        humid =  [pred["main"]["humidity"] for pred in previsions]       # humidité en %
        nuages = [pred["clouds"]["all"] for pred in previsions]          # couverture nuageuse en %
        vent = [pred["wind"]["speed"] for pred in previsions]            # vitesse du vent en m/s
        vent_max = [pred["wind"]["gust"] for pred in previsions]         # vitesse max du vent en m/s
        pops  = [pred.get("pop", 0) for pred in previsions]              # probabilité de précipitation [0-1]
        pluie = [pred.get("rain", {}).get("3h", 0) for pred in previsions]  # volume de pluie en mm
        neige = [pred.get("now", {}).get("3h", 0) for pred in previsions]  # volume de neige en mm

        # Retenir la moyenne des paramètres sur l'ensemble de la période (5 jours)
        weather_results.append({
            "city_id": row["city_id"],
            "temp": round(float(np.mean(temps)), 2), 
            "temp_feels": round(float(np.mean(temps_feels)), 2),
            "temp_min": round(float(np.mean(temps_min)), 2),
            "temp_max": round(float(np.mean(temps_max)), 2),
            "humidity": round(float(np.mean(humid)), 1),
            "clouds": round(float(np.mean(nuages)), 1),
            "wind": round(float(np.mean(vent)), 2),
            "wind_max": round(float(np.mean(vent_max)), 2),
            "rain_prob": round(float(np.mean(pops)), 2),
            "rain_volume": round(float(np.sum(pluie)), 1),
            "snow_volume": round(float(np.sum(neige)), 1),
        })
        print(f"{row['city']}")
    else:
        print(f"{row['city']} - code {response.status_code}")

# Merge des données météo avec les coordonnées GPS
weather_df = pd.DataFrame(weather_results)
if weather_df.empty:
    if not API_KEY:
        raise ValueError(
            "OPENWEATHER_API_KEY introuvable. Vérifiez que '.env' existe "
            "et contient la variable OPENWEATHER_API_KEY, ou exportez-la dans l'environnement."
        )
    else:
        raise ValueError(
            "Aucune donnée météo n'a été récupérée. Vérifiez que la clé OPENWEATHER_API_KEY est valide "
            "et que les requêtes vers OpenWeatherMap renvoient 200 (resp.status_code)."
        )
        
df_weather_final = pd.merge(
    df_villes,
    weather_df,
    on="city_id",
    how="left"
)
df_weather_final.to_csv(DATA/"villes_meteo_previsions.csv", index=False)

print(f"\nMeteo agrégée pour {len(df_weather_final)} villes")
display(df_weather_final.head())

Mont Saint Michel
St Malo
Bayeux
Le Havre
Rouen
Paris
Amiens
Lille
Strasbourg
Chateau du Haut Koenigsbourg
Colmar
Eguisheim
Besancon
Dijon
Annecy
Grenoble
Lyon
Gorges du Verdon
Bormes les Mimosas
Cassis
Marseille
Aix en Provence
Avignon
Uzes
Nimes
Aigues Mortes
Saintes Maries de la Mer
Collioure
Carcassonne
Ariege
Toulouse
Montauban
Biarritz
Bayonne
La Rochelle

Meteo agrégée pour 35 villes


,city_id,city,latitude,longitude,temp,temp_feels,temp_min,temp_max,humidity,clouds,wind,wind_max,rain_prob,rain_volume,snow_volume
0,1,Mont Saint Michel,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0
1,2,St Malo,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0
2,3,Bayeux,49.276462,-0.702474,18.18,18.15,18.18,18.38,80.5,83.1,4.00,7.03,0.18,21.0,0.0
3,4,Le Havre,49.493898,0.107973,18.97,19.03,18.97,19.05,80.8,79.0,4.72,6.26,0.19,17.7,0.0
4,5,Rouen,49.440459,1.093966,19.21,19.23,19.21,19.44,78.6,76.8,2.57,5.38,0.16,11.7,0.0


## 4. Scraping de Booking.com
`requests` est bloqué par Booking -> utilisation de Selenium.

Stratégie : scroll progressif pour déclencher le lazy-loading.

In [4]:
def scrape_hotels_selenium(row):
    """
    Scrape les hôtels d'une ville sur Booking.com via Selenium Chrome headless.

    Pourquoi pas requests ?
    -> Booking.com charge son contenu via JavaScript (rendu dynamique).
    -> requests ne récupère que le HTML statique : les cartes hôtels sont absentes.
       Selenium pilote un vrai navigateur qui execute le JS.

    Return: list[dict] {city, hotel_name, url, rating, description, ...}
    """
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

    # Configuration des options spécifiques à undetected-chromedriver
    # opts = uc.ChromeOptions()
    # opts.add_argument("--headless=new")  # Syntaxe headless moderne recommandée par uc
    # opts.add_argument("--window-size=1920,1080")
    
    # # Remarque : uc gère déjà un User-Agent réaliste, mais tu peux forcer celui-ci si besoin
    # opts.add_argument(
    #     "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    #     "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    # )

    # # Initialisation du driver undetected (uc télécharge et gère son propre driver sous le capot)
    # driver = uc.Chrome(options=opts)

    hotels = []
    today = date.today()
    checkin = today + timedelta(days=1)
    checkout = checkin + timedelta(days=5)
    # Recherche standard basée sur la ville avec la date d'arrivée et de départ
    url = (
        f"https://www.booking.com/searchresults.fr.html?"
        f"&latitude={row['latitude']}&longitude={row['longitude']}"
        f"&checkin={checkin.strftime('%Y-%m-%d')}"
        f"&checkout={checkout.strftime('%Y-%m-%d')}"
    )
    #f"ss={row["city"].replace(' ', '%20')}"
    
    try:
        driver.get(url)
        time.sleep(5)

        # Scroll progressif pour déclencher le lazy-loading des cartes hôtels
        for i in range(4):
            driver.execute_script(f"window.scrollTo(0, {(i+1)*1000});")
            time.sleep(random.uniform(2.0, 3.0))

        # Récupération des données
        raw_cards = []
        cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')[:15]
        for card in cards:
            try:
                # Extraction du nom et l'url depuis la carte
                name = card.find_element(
                    By.CSS_SELECTOR, '[data-testid="title"]'
                ).text.strip()
                hotel_url = card.find_element(
                    By.CSS_SELECTOR, '[data-testid="title-link"]'
                ).get_attribute("href")

                # Extraction du prix depuis la carte
                price = None
                try:
                    p_text = card.find_element(
                        By.CSS_SELECTOR,
                        '[data-testid="price-and-discounted-price"]',
                    ).text
                    clean = re.sub(r"[^\d]", "", p_text)
                    if clean:
                        price = float(clean)
                except:
                    pass

                # Extraction de la note
                rating = None
                try:
                    raw_score = card.find_element(
                        By.CSS_SELECTOR, '[data-testid="review-score"]'
                    ).text
                    m = re.search(r"(\d[,\.]\d)", raw_score)
                    if m:
                        rating = float(m.group(1).replace(",", "."))
                except:
                    pass

                # Description / type de chambre
                try:
                    desc = card.find_element(By.CSS_SELECTOR, '[data-testid="recommended-units"]').text.strip()
                except:
                    desc = "N/A"

                if name:
                    raw_cards.append(
                        {
                            "name": name,
                            "url": hotel_url,
                            "price": price,
                            "rating": rating,
                            "description": desc
                        }
                    )
            except:
                continue

        # Récupération des coordonnées gps depuis les url des hôtels
        for item in raw_cards:
            try:
                driver.get(item["url"])
                time.sleep(2)

                # Extraction Latitude / Longitude (Booking l'injecte dans l'attribut data-atlas-latlng)
                hotel_lat, hotel_lon = None, None
                try:
                    map_elem = driver.find_element(
                        By.CSS_SELECTOR,
                        "[data-atlas-latlng], #hotel_sidebar_static_map",
                    )
                    latlon_str = map_elem.get_attribute("data-atlas-latlng")

                    if latlon_str:
                        coords = latlon_str.split(",")
                        hotel_lat = float(coords[0])
                        hotel_lon = float(coords[1])
                except:
                    pass

                hotels.append(
                    {
                        "city_id": row["city_id"],
                        "city": row["city"],
                        "hotel_name": item['name'],
                        "hotel_latitude": hotel_lat,
                        "hotel_longitude": hotel_lon,
                        "hotel_price": item['price'],
                        "hotel_url": item['url'],
                        "hotel_rating": item['rating'],
                        "hotel_description": item['description'],
                    }
                )
            except:
                continue
    finally:
        driver.quit()

    return hotels

In [5]:
scrap_data  = []

for _, row in df_villes.iterrows():
    print(f"[{row['city_id']}/35] {row['city']}...")
    try:
        results = scrape_hotels_selenium(row)

        scrap_data.extend(results)

        print(f"  -> {len(results)} hôtels | total cumulé : {len(scrap_data)}")
    except Exception as e:
        print(f"Erreur : {e}")

    time.sleep(5)

df_hotels_raw = pd.DataFrame(scrap_data)
print(f"\nScraping terminé - {len(df_hotels_raw)} lignes")
df_hotels_raw.to_csv(DATA/"villes_booking_hotels.csv", index=False)
display(df_hotels_raw.head())

[1/35] Mont Saint Michel...
  -> 3 hôtels | total cumulé : 3
[2/35] St Malo...
  -> 3 hôtels | total cumulé : 6
[3/35] Bayeux...
  -> 3 hôtels | total cumulé : 9
[4/35] Le Havre...
  -> 3 hôtels | total cumulé : 12
[5/35] Rouen...
  -> 3 hôtels | total cumulé : 15
[6/35] Paris...
  -> 2 hôtels | total cumulé : 17
[7/35] Amiens...
  -> 3 hôtels | total cumulé : 20
[8/35] Lille...
  -> 2 hôtels | total cumulé : 22
[9/35] Strasbourg...
  -> 3 hôtels | total cumulé : 25
[10/35] Chateau du Haut Koenigsbourg...
  -> 3 hôtels | total cumulé : 28
[11/35] Colmar...
  -> 3 hôtels | total cumulé : 31
[12/35] Eguisheim...
  -> 3 hôtels | total cumulé : 34
[13/35] Besancon...
  -> 3 hôtels | total cumulé : 37
[14/35] Dijon...
  -> 3 hôtels | total cumulé : 40
[15/35] Annecy...
  -> 3 hôtels | total cumulé : 43
[16/35] Grenoble...
  -> 3 hôtels | total cumulé : 46
[17/35] Lyon...
  -> 3 hôtels | total cumulé : 49
[18/35] Gorges du Verdon...
  -> 3 hôtels | total cumulé : 52
[19/35] Bormes les Mimosa

,city_id,city,hotel_name,hotel_latitude,hotel_longitude,hotel_price,hotel_url,hotel_rating,hotel_description
0,1,Mont Saint Michel,Village de Gîtes de l'Anse de Moidrey,48.579988,-1.516531,532.0,https://www.booking.com/hotel/fr/village-de-gi...,7.8,Maison 1 Chambre (4 Personnes)\nMaison de vaca...
1,1,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,734.0,https://www.booking.com/hotel/fr/clerval.fr.ht...,9.3,Maison 1 Chambre\nMaison de vacances entière •...
2,1,Mont Saint Michel,Relaxing Holiday Home in Normandy,48.629603,-1.413223,1128.0,https://www.booking.com/hotel/fr/relaxing-holi...,NaN,Maison de Vacances 5 Chambres\nMaison de vacan...
3,2,St Malo,Hôtel France et Chateaubriand,48.651193,-2.023796,1713.0,https://www.booking.com/hotel/fr/de-france-et-...,8.1,Chambre Familiale Prestige\nPlusieurs types de...
4,2,St Malo,Le Pont Supérieur,48.647585,-2.024273,900.0,https://www.booking.com/hotel/fr/le-pont-super...,7.8,Appartement 3 Chambres\nAppartement entier • 3...


## 5. Data Lake : AWS S3
Clés AWS chargées depuis `.env`.

Regrouper toutes les informations `brutes` collectées dans un csv et le sauvegarder dans un data lake.

In [25]:
# --- Fusion météo + hôtels ---
df_final = pd.merge(df_hotels_raw, df_weather_final, on=["city_id", "city"], how="left")
df_final = df_final.rename(columns={'latitude': 'city_latitude', 'longitude': 'city_longitude'})
df_final.to_csv(DATA/"villes_raw_data.csv", index=False)

In [26]:
pd.options.display.max_columns = None
df_final

,city_id,city,hotel_name,hotel_latitude,hotel_longitude,hotel_price,hotel_url,hotel_rating,hotel_description,city_latitude,city_longitude,temp,temp_feels,temp_min,temp_max,humidity,clouds,wind,wind_max,rain_prob,rain_volume,snow_volume
0,1,Mont Saint Michel,Village de Gîtes de l'Anse de Moidrey,48.579988,-1.516531,532.0,https://www.booking.com/hotel/fr/village-de-gi...,7.8,Maison 1 Chambre (4 Personnes)\nMaison de vaca...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0
1,1,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,734.0,https://www.booking.com/hotel/fr/clerval.fr.ht...,9.3,Maison 1 Chambre\nMaison de vacances entière •...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0
2,1,Mont Saint Michel,Relaxing Holiday Home in Normandy,48.629603,-1.413223,1128.0,https://www.booking.com/hotel/fr/relaxing-holi...,NaN,Maison de Vacances 5 Chambres\nMaison de vacan...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0
3,2,St Malo,Hôtel France et Chateaubriand,48.651193,-2.023796,1713.0,https://www.booking.com/hotel/fr/de-france-et-...,8.1,Chambre Familiale Prestige\nPlusieurs types de...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0
4,2,St Malo,Le Pont Supérieur,48.647585,-2.024273,900.0,https://www.booking.com/hotel/fr/le-pont-super...,7.8,Appartement 3 Chambres\nAppartement entier • 3...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,34,Bayonne,Premiere Classe Bayonne,43.490490,-1.452035,389.0,https://www.booking.com/hotel/fr/premiere-clas...,7.0,Chambre Lit Queen-Size Standard avec 1 Lit Sup...,43.494514,-1.473666,22.62,23.01,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0
100,34,Bayonne,hotelF1 Bayonne,43.489623,-1.451075,257.0,https://www.booking.com/hotel/fr/hotelf1-bayon...,5.3,Chambre Side-Car\nAnnulation gratuite,43.494514,-1.473666,22.62,23.01,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0
101,35,La Rochelle,"Studio Hyper centre, quartier historique",46.163420,-1.153443,432.0,https://www.booking.com/hotel/fr/studio-hyper-...,5.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,-1.151595,20.85,21.00,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0
102,35,La Rochelle,Charmant T2 place de Verdun,46.163420,-1.153443,791.0,https://www.booking.com/hotel/fr/charmant-t2-p...,8.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,-1.151595,20.85,21.00,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0


In [ ]:
#  Les clés et paramètres AWS sont chargés dans des variables d'environnement (à partir du .env)
ACCESS_KEY  = os.getenv("AWS_ACCESS_KEY_ID")
SECRET_KEY  = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
REGION      = os.getenv("AWS_REGION", "eu-west-3")

# Session AWS authentifiée
session   = boto3.Session(aws_access_key_id=ACCESS_KEY,
                          aws_secret_access_key=SECRET_KEY,
                          region_name=REGION)
s3_client = session.client("s3")

# Création du bucket S3 s'il n'existe pas encore
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket déjà existant.")
except:
    s3_client.create_bucket(Bucket=BUCKET_NAME,
                     CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"Bucket créé.")

# Upload du fichier .csv contenant les données brutes collectées dans un bucket S3
filename = "kayak_raw_data.csv"
local_path = DATA/"villes_raw_data.csv"
if local_path.exists():
    s3.Bucket(BUCKET_NAME).upload_file(str(local_path), filename)
    print(f"Fichier envoyé")
else:
    print(f"Fichier non trouvé localement")

Bucket déjà existant.
Fichier envoyé


## 6. Extract & Transform
**Nettoyage :** suppression des lignes sans note, deduplication par `(hotel_name, city)`.
**Géocodage des hôtels :** le cahier des charges demande la latitude / longitude de chaque hôtel. On interroge Nominatim sur "nom hôtel, ville, France" ; en cas d'echec, on retombe sur les coordonnées du centre-ville (fallback).
**Score de voyage :** chaque critère (température, note hôtelière, faible pluie) est normalisée entre 0 et 1, puis pondère. Cette méthode est plus rigoureuse que de mélanger des échelles differentes.

In [ ]:
# --- EXTRACT ---
# Récupérer le fichier .csv depuis le S3 --> toutes les données brutes qui ont été collectées
csv_obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=filename)
df_raw = pd.read_csv(csv_obj["Body"])

In [ ]:
# --- TRANSFORM ---
# Nettoyage
df_raw = df_raw.drop_duplicates(subset=["city", "hotel_name"]).reset_index(drop=True)  # suppression des doublons
df_raw["hotel_description"] = df_raw["hotel_description"].fillna("N/A")                # remplacer les descriptions vides par Not Available "N/A"
df_raw = df_raw.dropna(subset=["hotel_rating"]).reset_index(drop=True)                 # garder uniquement les hôtels notés

In [ ]:
# Construire un modèle de notation méteo
def get_thermal_rating(temp: float) -> int:
    """Note de 0 à 10 pour le confort thermique (°C)."""
    if 22 <= temp <= 25:
        return 10
    elif 19 <= temp < 22 or 25 < temp <= 28:
        return 9
    elif 16 <= temp < 19 or 28 < temp <= 31:
        return 7
    elif 10 <= temp < 16 or 31 < temp <= 35:
        return 4
    elif 0 <= temp < 10 or 35 < temp <= 38:
        return 2
    return 0

def get_precip_rating(precip_mm: float) -> int:
    """Note de 0 à 10 pour les précipitations (mm)."""
    if precip_mm == 0:
        return 10
    elif precip_mm <= 2.0:
        return 8
    elif precip_mm <= 5.0:
        return 6
    elif precip_mm <= 12.0:
        return 3
    return 0

def get_cloud_rating(cloud_pct: float) -> int:
    """Note de 0 à 10 pour la couverture nuageuse (%)."""
    if cloud_pct <= 20:
        return 10
    elif cloud_pct <= 40:
        return 9
    elif cloud_pct <= 60:
        return 7
    elif cloud_pct <= 80:
        return 5
    return 2

def get_wind_rating(wind_kmh: float) -> int:
    """Note de 0 à 10 pour le vent (km/h)."""
    if wind_kmh <= 10:
        return 10
    elif wind_kmh <= 20:
        return 9
    elif wind_kmh <= 30:
        return 6
    elif wind_kmh <= 40:
        return 3
    return 0

def calculate_hci_urban(temp_feels_like: float, precip_mm: float, cloud_pct: float, wind_kmh: float) -> float:
    """
    Calcule le score HCI Urban (0-100).
    """
    r_thermal = get_thermal_rating(temp_feels_like)
    r_precip = get_precip_rating(precip_mm)
    r_cloud = get_cloud_rating(cloud_pct)
    r_wind = get_wind_rating(wind_kmh)
    
    # Formule HCI : (4 * Thermal) + (3 * Precip) + (2 * Cloud) + (1 * Wind)
    hci_score = (2 * r_thermal) + (4 * r_precip) + (3 * r_cloud) + (1 * r_wind)
    
    # Facteur veto : si plus de 15 mm de pluie ou tempête (> 50 km/h)
    # if precip_mm > 15 or wind_kmh > 50:
    #     hci_score *= 0.2
        
    return round(hci_score, 1)

In [ ]:
# Enrichir le dataset final avec le score météo de chaque destination
df_cleaned = df_raw.copy()
df_cleaned["score_meteo"] = 0
for index, row in df_raw.iterrows():
    df_cleaned.loc[index, "score_meteo"] = calculate_hci_urban(row["temp_feels"], row["rain_volume"], row["clouds"], row["wind"])

In [111]:
df_cleaned

,city_id,city,hotel_name,hotel_latitude,hotel_longitude,hotel_price,hotel_url,hotel_rating,hotel_description,city_latitude,city_longitude,temp,temp_feels,temp_min,temp_max,humidity,clouds,wind,wind_max,rain_prob,rain_volume,snow_volume,score_meteo
0,1,Mont Saint Michel,Village de Gîtes de l'Anse de Moidrey,48.579988,-1.516531,532.0,https://www.booking.com/hotel/fr/village-de-gi...,7.8,Maison 1 Chambre (4 Personnes)\nMaison de vaca...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
1,1,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,734.0,https://www.booking.com/hotel/fr/clerval.fr.ht...,9.3,Maison 1 Chambre\nMaison de vacances entière •...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
2,2,St Malo,Hôtel France et Chateaubriand,48.651193,-2.023796,1713.0,https://www.booking.com/hotel/fr/de-france-et-...,8.1,Chambre Familiale Prestige\nPlusieurs types de...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
3,2,St Malo,Le Pont Supérieur,48.647585,-2.024273,900.0,https://www.booking.com/hotel/fr/le-pont-super...,7.8,Appartement 3 Chambres\nAppartement entier • 3...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
4,2,St Malo,Le Cap-Horn - dans l'Intra-Muros de St Malo,48.647369,-2.024488,690.0,https://www.booking.com/hotel/fr/le-cap-horn-d...,9.0,Appartement 2 Chambres\nAppartement entier • 2...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,34,Bayonne,Premiere Classe Bayonne,43.490490,-1.452035,389.0,https://www.booking.com/hotel/fr/premiere-clas...,7.0,Chambre Lit Queen-Size Standard avec 1 Lit Sup...,43.494514,-1.473666,22.62,23.01,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0,77
89,34,Bayonne,hotelF1 Bayonne,43.489623,-1.451075,257.0,https://www.booking.com/hotel/fr/hotelf1-bayon...,5.3,Chambre Side-Car\nAnnulation gratuite,43.494514,-1.473666,22.62,23.01,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0,77
90,35,La Rochelle,"Studio Hyper centre, quartier historique",46.163420,-1.153443,432.0,https://www.booking.com/hotel/fr/studio-hyper-...,5.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,-1.151595,20.85,21.00,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0,55
91,35,La Rochelle,Charmant T2 place de Verdun,46.163420,-1.153443,791.0,https://www.booking.com/hotel/fr/charmant-t2-p...,8.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,-1.151595,20.85,21.00,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0,55


In [ ]:
df_cleaned.to_csv(DATA/"kayak_final_enriched_data.csv", index=False)

print(f"Dataset final : {df_cleaned.shape}")
display(df_cleaned.head(10))

Dataset final : (93, 23)


,city_id,city,hotel_name,hotel_latitude,hotel_longitude,hotel_price,hotel_url,hotel_rating,hotel_description,city_latitude,city_longitude,temp,temp_feels,temp_min,temp_max,humidity,clouds,wind,wind_max,rain_prob,rain_volume,snow_volume,score_meteo
0,1,Mont Saint Michel,Village de Gîtes de l'Anse de Moidrey,48.579988,-1.516531,532.0,https://www.booking.com/hotel/fr/village-de-gi...,7.8,Maison 1 Chambre (4 Personnes)\nMaison de vaca...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
1,1,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,734.0,https://www.booking.com/hotel/fr/clerval.fr.ht...,9.3,Maison 1 Chambre\nMaison de vacances entière •...,48.635954,-1.511460,18.48,18.48,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
2,2,St Malo,Hôtel France et Chateaubriand,48.651193,-2.023796,1713.0,https://www.booking.com/hotel/fr/de-france-et-...,8.1,Chambre Familiale Prestige\nPlusieurs types de...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
3,2,St Malo,Le Pont Supérieur,48.647585,-2.024273,900.0,https://www.booking.com/hotel/fr/le-pont-super...,7.8,Appartement 3 Chambres\nAppartement entier • 3...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
4,2,St Malo,Le Cap-Horn - dans l'Intra-Muros de St Malo,48.647369,-2.024488,690.0,https://www.booking.com/hotel/fr/le-cap-horn-d...,9.0,Appartement 2 Chambres\nAppartement entier • 2...,48.649518,-2.026041,18.41,18.42,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
5,3,Bayeux,COSY HOUSE jardin parking gratuit,49.279860,-0.715986,632.0,https://www.booking.com/hotel/fr/cosy-house.fr...,8.0,Maison de Vacances\nMaison de vacances entière...,49.276462,-0.702474,18.18,18.15,18.18,18.38,80.5,83.1,4.00,7.03,0.18,21.0,0.0,30
6,3,Bayeux,Les Libérateurs - Collection La 21ème Planche,49.272479,-0.718725,921.0,https://www.booking.com/hotel/fr/les-liberateu...,9.5,Maison 3 Chambres\nMaison de vacances entière ...,49.276462,-0.702474,18.18,18.15,18.18,18.38,80.5,83.1,4.00,7.03,0.18,21.0,0.0,30
7,3,Bayeux,Novotel Bayeux,49.281769,-0.718489,1001.0,https://www.booking.com/hotel/fr/novotel.fr.ht...,8.1,Chambre Classique Â avec Lit Double et Canapé-...,49.276462,-0.702474,18.18,18.15,18.18,18.38,80.5,83.1,4.00,7.03,0.18,21.0,0.0,30
8,4,Le Havre,Le Perret -Centre -Authentique -UNESCO,49.489111,0.106701,379.0,https://www.booking.com/hotel/fr/le-perret-cen...,8.3,Appartement 1 Chambre\nAppartement entier • 1 ...,49.493898,0.107973,18.97,19.03,18.97,19.05,80.8,79.0,4.72,6.26,0.19,17.7,0.0,43
9,4,Le Havre,Charmant studio en centre-ville / proche gare,49.493310,0.115493,232.0,https://www.booking.com/hotel/fr/charmant-stud...,5.2,Appartement 1 Chambre\nAppartement entier • 1 ...,49.493898,0.107973,18.97,19.03,18.97,19.05,80.8,79.0,4.72,6.26,0.19,17.7,0.0,43


In [2]:
df_cleaned = pd.read_csv(DATA/"kayak_final_enriched_data.csv")

In [15]:
df_cleaned

,city_id,city,hotel_name,hotel_latitude,hotel_longitude,hotel_price,hotel_url,hotel_rating,hotel_description,city_latitude,...,temp_min,temp_max,humidity,clouds,wind,wind_max,rain_prob,rain_volume,snow_volume,score_meteo
0,1,Mont Saint Michel,Village de Gîtes de l'Anse de Moidrey,48.579988,-1.516531,532.0,https://www.booking.com/hotel/fr/village-de-gi...,7.8,Maison 1 Chambre (4 Personnes)\nMaison de vaca...,48.635954,...,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
1,1,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,734.0,https://www.booking.com/hotel/fr/clerval.fr.ht...,9.3,Maison 1 Chambre\nMaison de vacances entière •...,48.635954,...,18.48,18.58,80.2,77.0,4.08,7.08,0.19,18.1,0.0,39
2,2,St Malo,Hôtel France et Chateaubriand,48.651193,-2.023796,1713.0,https://www.booking.com/hotel/fr/de-france-et-...,8.1,Chambre Familiale Prestige\nPlusieurs types de...,48.649518,...,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
3,2,St Malo,Le Pont Supérieur,48.647585,-2.024273,900.0,https://www.booking.com/hotel/fr/le-pont-super...,7.8,Appartement 3 Chambres\nAppartement entier • 3...,48.649518,...,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
4,2,St Malo,Le Cap-Horn - dans l'Intra-Muros de St Malo,48.647369,-2.024488,690.0,https://www.booking.com/hotel/fr/le-cap-horn-d...,9.0,Appartement 2 Chambres\nAppartement entier • 2...,48.649518,...,18.41,18.53,80.8,79.7,4.71,7.51,0.16,15.4,0.0,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,34,Bayonne,Premiere Classe Bayonne,43.490490,-1.452035,389.0,https://www.booking.com/hotel/fr/premiere-clas...,7.0,Chambre Lit Queen-Size Standard avec 1 Lit Sup...,43.494514,...,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0,77
89,34,Bayonne,hotelF1 Bayonne,43.489623,-1.451075,257.0,https://www.booking.com/hotel/fr/hotelf1-bayon...,5.3,Chambre Side-Car\nAnnulation gratuite,43.494514,...,22.62,22.74,75.9,68.1,4.13,5.67,0.04,0.8,0.0,77
90,35,La Rochelle,"Studio Hyper centre, quartier historique",46.163420,-1.153443,432.0,https://www.booking.com/hotel/fr/studio-hyper-...,5.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,...,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0,55
91,35,La Rochelle,Charmant T2 place de Verdun,46.163420,-1.153443,791.0,https://www.booking.com/hotel/fr/charmant-t2-p...,8.7,Appartement 1 Chambre\nAppartement entier • 1 ...,46.159732,...,20.85,21.05,76.9,62.3,3.91,6.28,0.08,6.8,0.0,55


## 7. Load --> Data Warehouse(AWS Redshift)
Clés chargées depuis `.env`.

In [ ]:
#  Les paramètres AWS Redshift sont chargés dans des variables d'environnement (à partir du .env)
DB_USER     = os.getenv("REDS_USERNAME")
DB_PASSWORD = os.getenv("REDS_PASSWORD")
DB_HOST     = os.getenv("REDS_HOSTNAME")              # Endpoint du workgroup de redshift
DB_PORT     = os.getenv("REDS_PORT", "5439")          # Port de AWS redshift
DB_NAME     = os.getenv("REDS_DB_NAME", "postgres")   # Nom de la base de données

# Connexion via SQLAlchemy
connection_string = f"redshift+redshift_connector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    # Envoi de la table
    with engine.begin() as conn:
        df_cleaned.to_sql(
            "kayak_destinations",
            con=conn,
            if_exists="replace",
            index=False,
            dtype={
                'hotel_url': VARCHAR(2048),
                'hotel_description': VARCHAR(65535),
                'hotel_name': VARCHAR(2048),
                'city': VARCHAR(256)
            }
        )
    # Fermer le pool de connexions pour libérer Redshift
    engine.dispose()
    print(f"Table injectée")
except Exception as e:
    print(f"Erreur Redshift : {e}")

Table injectée


## 8. Visualisation : Cartes interactives Plotly
### Carte 1 : Top 5 destinations

In [ ]:
# --- Récuépérer le Top 5 des destinations ---
with engine.connect() as conn:
    stmt = text("SELECT city, city_latitude, city_longitude, score_meteo FROM kayak_destinations "
                "GROUP BY city, city_latitude, city_longitude, score_meteo "
                "ORDER BY score_meteo DESC "
                "LIMIT 5")

    df_top_destinations = conn.execute(stmt).fetchall()

engine.dispose()

In [64]:
df_top_destinations

,city,city_latitude,city_longitude,score_meteo
0,Besancon,47.238022,6.024362,95
1,Strasbourg,48.584614,7.750713,91
2,Chateau du Haut Koenigsbourg,48.249382,7.343941,91
3,Colmar,48.077752,7.357964,91
4,Eguisheim,48.044797,7.307962,91


In [69]:
fig = px.scatter_map(
    df_top_destinations,
    lat="city_latitude", 
    lon="city_longitude",
    hover_name="city",
    hover_data={
        "city_latitude": False, 
        "city_longitude": False,
        "score_meteo": True,
    },
    color="score_meteo",
    size="score_meteo",
    color_continuous_scale="viridis",
    zoom=1,
    height=700,
    title="Top 5 Destinations"
)
fig.show()

### Carte 2 : Top 20 hôtels les mieux notés

In [ ]:
# --- Récuépérer le Top 20 des hotels ---
with engine.connect() as conn:
    stmt = text("SELECT city, hotel_name, hotel_latitude, hotel_longitude, score_meteo, hotel_rating FROM kayak_destinations "
                "ORDER BY hotel_rating DESC "
                "LIMIT 20")

    df_top_hotels = conn.execute(stmt).fetchall()

engine.dispose()

In [72]:
df_top_hotels

,city,hotel_name,hotel_latitude,hotel_longitude,score_meteo,hotel_rating
0,Eguisheim,Appartement Le Belarya,48.073433,7.289648,91,9.7
1,Bormes les Mimosas,T2 classé 3 étoiles avec piscine et climatisat...,43.143542,6.342481,89,9.7
2,Uzes,BEL APPARTEMENT SPACIEUX CENTRE VILLE UZES T3,44.013700,4.417993,81,9.6
3,Ariege,Gite du Pic noir,42.964024,1.242898,77,9.6
4,Annecy,Lac arte postale- Trois climatisations murales...,45.894600,6.132588,91,9.5
5,Bayeux,Les Libérateurs - Collection La 21ème Planche,49.272479,-0.718725,30,9.5
6,Grenoble,chambre individuelle très confortable avec verrou,45.180592,5.737808,89,9.5
7,Chateau du Haut Koenigsbourg,"Les Gîtes de la tour, appartement ""le Vignoble""",48.231101,7.370074,91,9.4
8,Mont Saint Michel,B&B Clerval - Maison d'hôtes et Gîtes,48.698367,-1.492418,39,9.3
9,Saintes Maries de la Mer,L Abri-côtier Appartements,43.454350,4.427723,75,9.3


In [76]:
# --- Carte interactive : Top 20 hôtels (coordonnées propres à chaque hôtel) ---
fig2 = px.scatter_map(
    df_top_hotels,
    lat="hotel_latitude", 
    lon="hotel_longitude",
    hover_name="hotel_name",
    hover_data={
        "hotel_latitude": False, 
        "hotel_longitude": False,
        "city": True,
        "hotel_rating": ":.1f"
    },
    color="score_meteo",
    size="hotel_rating",
    color_continuous_scale=px.colors.sequential.Viridis,
    zoom=5, 
    map_style="open-street-map",
    title="<b>Top 20 Hôtels les mieux notés en France</b>"
)
fig2.update_layout(margin={"r": 10, "t": 50, "l": 10, "b": 10})
fig2.show()

## 9. Conformité au RGPD

Le processus de collecte a été concu dans le respect du Réglement Général sur la Protection des Données.

- **Absence de données personnelles.** Le projet ne collecte aucune donnée relative à une personne physique identifiée ou identifiable. Les données traitées sont des informations publiques et non nominatives : coordonnées geographiques de villes, prévisions météorologiques, et fiches d'établissements hôteliers (nom commercial, note agrégée, description, lien). Les avis individuels et les noms d'utilisateurs ne sont ni collectés ni stockés.

- **Sources et conditions d'utilisation.** Les données météo proviennent d'une API publique utilisée dans le cadre de ses conditions. Le scraping de Booking.com se limite à des informations publiquement affichées, avec des cadences d'appel raisonnables (pauses entre requêtes) afin de ne pas surcharger le service.

- **Stockage maitrisé.** Les données sont hébergées sur une infrastructure AWS dont l'accés est restreint (identifiants dédies, groupe de sécurité RDS limitant les adresses IP autorisées).